In [1]:
import json
from pathlib import Path
import polars as pl
from IPython.display import Markdown
from polars import DataFrame, LazyFrame

In [2]:
from pan26_genai_detection.evaluator.evaluator import c_at_1, auc, f1, f05u, brier_score

def evaluate_all(y_true: pl.DataFrame, y_pred: pl.DataFrame):
    y_true = y_true.to_numpy()
    y_pred = y_pred.to_numpy()
    return pl.from_dict({
        'roc-auc': auc(y_true, y_pred),
        'brier': brier_score(y_true, y_pred),
        'c@1': c_at_1(y_true, y_pred),
        'f1': f1(y_true, y_pred),
        'f05u': f05u(y_true, y_pred),
    }).with_columns(
        pl.mean_horizontal(pl.all()).alias('mean'),
        pl.lit((y_pred[y_true == 0] >= 0.5).sum() / (y_true == 0).sum()).alias('fpr'),
        pl.lit((y_pred[y_true == 1] < 0.5).sum() / (y_true == 1).sum()).alias('fnr'),
    )

In [3]:
def build_truth_map(truth_file: Path, id_map_file: Path = None):
    truth_df = pl.read_ndjson(truth_file)
    if not id_map_file:
        return truth_df.with_columns(pl.lit(None).alias('orig_id'))
    id_map_df = pl.read_ndjson(id_map_file)
    return truth_df.join(id_map_df, on='id', how='left')

def load_run_files(rundir: Path):
    meta_df = pl.read_ndjson(rundir / 'metadata.jsonl')
    runs_df = []
    run_ids = meta_df.select('run_id').unique().to_series()

    # Leave no stray runs behind
    assert len(run_ids) == len(list(rundir.joinpath('outputs-flat').glob('2026-*')))

    for rid in run_ids:
        runfile = rundir / 'outputs-flat' / rid
        for l in runfile.open('r'):
            j = json.loads(l)
            j['run_id'] = rid
            runs_df.append(j)

    df = pl.from_records(runs_df)
    df = df.join(meta_df.select(['run_id', 'team', 'software', 'dataset']), on='run_id', how='left')
    return df

def join_runs_and_truth(runs_df: pl.DataFrame, meta_df: pl.DataFrame):
    df = runs_df.join(meta_df, on='id', how='left')
    df = df.rename({'label': 'label_pred', 'label_right': 'label_truth'})

    # Sanity check
    assert df['label_truth'].is_not_null().all()
    assert df['label_pred'].is_not_null().all()

    # Reorder columns and sort
    col_order = ['team', 'software', 'run_id', 'id', 'orig_id', 'model', 'source', 'label_pred', 'label_truth', 'dataset']
    df = df.select(col_order)
    df = df.sort(by=['team', 'software', 'run_id', 'model', 'label_truth'])
    return df

def load_run_truth_and_meta(base_path: Path, truth_file: Path, id_map_file: Path | None = None):
    truth_map_pan26 = build_truth_map(truth_file, id_map_file)
    runs_pan26 = load_run_files(base_path)
    return join_runs_and_truth(runs_pan26, truth_map_pan26)

In [4]:
BASE_PATH = Path('../data')
BASE_EVAL_PATH = BASE_PATH / 'task-eval'
BASE_RUN_PATH = BASE_EVAL_PATH / 'tira-runs'

In [5]:
pan25_df = load_run_truth_and_meta(BASE_RUN_PATH / 'pan25',
                                   BASE_PATH / 'splits' / 'pan25-voight-kampff-test-truth.jsonl')
pan25_df

team,software,run_id,id,orig_id,model,source,label_pred,label_truth,dataset
str,str,str,str,null,str,str,f64,i64,str
"""a-a-detection""","""navy-iceberg""","""2026-05-22-05-17-10""","""9c4d7515-347d-52df-b18f-60e2b4…",null,"""deepseek-r1-distill-qwen-32b""","""riddell-juola-o1-deepseek""",0.721133,1,"""pan25-generative-ai-detection-…"
"""a-a-detection""","""navy-iceberg""","""2026-05-22-05-17-10""","""85ee111e-7793-561d-adeb-c6005d…",null,"""deepseek-r1-distill-qwen-32b""","""riddell-juola-o1-deepseek""",0.719153,1,"""pan25-generative-ai-detection-…"
"""a-a-detection""","""navy-iceberg""","""2026-05-22-05-17-10""","""d28287f4-4b7c-5e40-86fc-d80048…",null,"""deepseek-r1-distill-qwen-32b""","""brennan-greenstadt""",0.72062,1,"""pan25-generative-ai-detection-…"
"""a-a-detection""","""navy-iceberg""","""2026-05-22-05-17-10""","""8116d325-14ce-51f5-a8a8-e8154a…",null,"""deepseek-r1-distill-qwen-32b""","""riddell-juola-o1-deepseek""",0.721147,1,"""pan25-generative-ai-detection-…"
"""a-a-detection""","""navy-iceberg""","""2026-05-22-05-17-10""","""914875d8-dbbb-506a-8af6-12701d…",null,"""deepseek-r1-distill-qwen-32b""","""brennan-greenstadt""",0.72212,1,"""pan25-generative-ai-detection-…"
…,…,…,…,…,…,…,…,…,…
"""zhengqiaozeng""","""three""","""2026-05-10-22-08-31""","""025b0308-dffc-5192-9fbc-63ee2f…",null,"""text-bison-002""","""pan24-test""",0.999997,1,"""pan25-generative-ai-detection-…"
"""zhengqiaozeng""","""three""","""2026-05-10-22-08-31""","""cd7f6528-aa55-5481-b16b-98a2dd…",null,"""text-bison-002""","""pan24-test""",0.999997,1,"""pan25-generative-ai-detection-…"
"""zhengqiaozeng""","""three""","""2026-05-10-22-08-31""","""25f67501-5396-5d44-9681-921803…",null,"""text-bison-002""","""pan24-test""",0.999997,1,"""pan25-generative-ai-detection-…"


In [6]:
pan26_df = load_run_truth_and_meta(BASE_RUN_PATH / 'pan26',
                                   BASE_PATH / 'splits' / 'pan26-voight-kampff-test-truth.jsonl',
                                   BASE_PATH / 'sampled' / 'pan26-voight-kampff-orig-ids.jsonl')
pan26_df

team,software,run_id,id,orig_id,model,source,label_pred,label_truth,dataset
str,str,str,str,str,str,str,f64,i64,str
"""a-a-detection""","""navy-iceberg""","""2026-05-22-00-36-05""","""a6d5ce11-948f-5935-9319-ee4509…","""gpt-4o/nytimes-writing-prompts…","""gpt-4o""","""nytimes-writing-prompts""",0.719311,1,"""pan26-generative-ai-detection-…"
"""a-a-detection""","""navy-iceberg""","""2026-05-22-00-36-05""","""233b35a4-986f-512b-8152-b09bb5…","""gpt-4o/nytimes-writing-prompts…","""gpt-4o""","""nytimes-writing-prompts""",0.721067,1,"""pan26-generative-ai-detection-…"
"""a-a-detection""","""navy-iceberg""","""2026-05-22-00-36-05""","""7b7b16a5-27a2-572a-a6c6-8a920d…","""gpt-4o/nytimes-writing-prompts…","""gpt-4o""","""nytimes-writing-prompts""",0.721603,1,"""pan26-generative-ai-detection-…"
"""a-a-detection""","""navy-iceberg""","""2026-05-22-00-36-05""","""51ce6717-03c0-5a37-ba2e-95f17a…","""gpt-4o/nytimes-writing-prompts…","""gpt-4o""","""nytimes-writing-prompts""",0.719695,1,"""pan26-generative-ai-detection-…"
"""a-a-detection""","""navy-iceberg""","""2026-05-22-00-36-05""","""a2335f76-1cc4-59e7-8efb-d7c49d…","""gpt-4o/nytimes-writing-prompts…","""gpt-4o""","""nytimes-writing-prompts""",0.718517,1,"""pan26-generative-ai-detection-…"
…,…,…,…,…,…,…,…,…,…
"""zhengqiaozeng""","""three""","""2026-05-08-05-27-50""","""5a5bfc46-426f-512d-8ee8-a73782…","""nytimes-writing-prompts-transl…","""nytimes-writing-prompts-transl…","""nytimes-writing-prompts-transl…",0.00016,1,"""pan26-generative-ai-detection-…"
"""zhengqiaozeng""","""three""","""2026-05-08-05-27-50""","""267951f0-5577-5892-aaf5-33282a…","""nytimes-writing-prompts-transl…","""nytimes-writing-prompts-transl…","""nytimes-writing-prompts-transl…",0.003989,1,"""pan26-generative-ai-detection-…"
"""zhengqiaozeng""","""three""","""2026-05-08-05-27-50""","""09030f49-bcf1-52a3-b090-215bb9…","""nytimes-writing-prompts-transl…","""nytimes-writing-prompts-transl…","""nytimes-writing-prompts-transl…",0.0002,1,"""pan26-generative-ai-detection-…"


In [7]:
def get_results_per_run(df: pl.DataFrame, drop_run_and_ds=False):
    group_cols = ['team', 'software', 'run_id', 'dataset']

    def evaluate_group(key, group_df):
        metrics = evaluate_all(
            group_df.select('label_truth'),
            group_df.select('label_pred'),
        )
        keys = pl.DataFrame([dict(zip(group_cols, key))])
        return pl.concat([keys, metrics], how='horizontal')

    df = pl.concat(
        evaluate_group(key, group_df)
        for key, group_df in df.partition_by(
            group_cols,
            as_dict=True,
            maintain_order=True,
        ).items()
    ).sort(by='mean', descending=True)
    if drop_run_and_ds:
        df = df.drop(['run_id', 'dataset'])
    return df

def get_best_by_team(result_per_run_df: pl.DataFrame, with_baselines=True):
    best_by_team = result_per_run_df.filter(pl.col('team') != 'baseline').unique(subset='team', keep='first', maintain_order=True)
    if not with_baselines:
        return best_by_team

    baseline_runs = result_per_run_df.filter(pl.col('team') == 'baseline')
    return pl.concat([best_by_team, baseline_runs]).sort('mean', descending=True)

In [8]:
results_per_run25 = get_results_per_run(pan25_df)
results_per_run26 = get_results_per_run(pan26_df)

In [9]:
display(Markdown('# PAN\'25 Dataset:'))
with pl.Config(float_precision=3, tbl_rows=-1):
    display(get_best_by_team(results_per_run25).drop(['dataset', 'run_id']))

# PAN'25 Dataset:

team,software,roc-auc,brier,c@1,f1,f05u,mean,fpr,fnr
str,str,f64,f64,f64,f64,f64,f64,f64,f64
"""mdok""","""mdok""",0.997,0.988,0.986,0.990,0.994,0.991,0.006,0.017
"""plagiarism-detector-com-reg2""","""alternating-contingency""",0.999,0.986,0.983,0.981,0.992,0.988,0.001,0.024
"""turingteam""","""marinated-edge""",0.995,0.979,0.980,0.983,0.991,0.986,0.010,0.025
"""dsgt-pan-26""","""relaxed-copy""",0.998,0.977,0.972,0.979,0.990,0.983,0.005,0.038
"""suspiciously-coherent""","""lead-log""",0.998,0.980,0.970,0.979,0.967,0.979,0.098,0.001
"""styloch2026""","""instant-tartar""",0.988,0.967,0.965,0.975,0.989,0.977,0.003,0.048
"""sst""","""DeBERTa-V3-large-individual""",0.987,0.969,0.965,0.975,0.987,0.977,0.010,0.045
"""ak-ys""","""Qwen3-0.6B-fine-tuned (dockerf…",0.991,0.957,0.949,0.962,0.982,0.968,0.009,0.069
"""zhengqiaozeng""","""three""",0.978,0.931,0.929,0.946,0.976,0.952,0.006,0.099


In [10]:
display(Markdown('# PAN\'26 Dataset:'))

best_by_team26 = get_best_by_team(results_per_run26)
leaderboard26 = best_by_team26.drop(['dataset', 'run_id'])

# xnoinspection PyTypeChecker
zero_knowledge_pan26 = pl.DataFrame({
    "team": "baseline",
    "software": "baseline-zero-knowledge",
    "roc-auc": 0.5,
    "brier": 0.749,
    "c@1": 0.749,
    "f1": 0.856,
    "f05u": 0.788,
    "mean": 0.728,
    "fpr": 1.0,
    "fnr": 0.0,
})

leaderboard26 = (
    pl
    .concat([leaderboard26, zero_knowledge_pan26])
    .sort('mean', descending=True)
    .with_columns([
        pl.col("team").replace("mdok", "baseline"),
        pl.col("software").replace("mdok", "baseline-mdok"),
    ])
)

leaderboard26.to_pandas().to_html(BASE_EVAL_PATH / 'pan26-leaderboard.html', index=True, float_format=lambda x: f"{x:.3f}")

with pl.Config(float_precision=3, tbl_rows=-1):
    display(leaderboard26)

# PAN'26 Dataset:

team,software,roc-auc,brier,c@1,f1,f05u,mean,fpr,fnr
str,str,f64,f64,f64,f64,f64,f64,f64,f64
"""suspiciously-coherent""","""lead-log""",0.989,0.965,0.965,0.976,0.980,0.975,0.053,0.029
"""sst""","""ModernBERT-large-BNN""",0.993,0.970,0.962,0.975,0.969,0.974,0.108,0.014
"""baseline""","""baseline-mdok""",0.995,0.951,0.946,0.963,0.984,0.968,0.002,0.072
"""plagiarism-detector-com-reg2""","""alternating-contingency""",0.996,0.941,0.935,0.899,0.957,0.946,0.000,0.072
"""dsgt-pan-26""","""relaxed-copy""",0.981,0.885,0.873,0.907,0.961,0.921,0.000,0.170
"""ydong""","""arid-winch""",0.930,0.915,0.886,0.928,0.896,0.911,0.420,0.014
"""turingteam""","""marinated-edge""",0.957,0.882,0.846,0.851,0.925,0.892,0.053,0.179
"""writerslogic-inc""","""large-rank""",0.891,0.891,0.853,0.902,0.899,0.887,0.312,0.092
"""a-a-detection""","""navy-iceberg""",0.852,0.848,0.826,0.885,0.880,0.858,0.375,0.106


In [11]:
pan26_df_filtered_best = pan26_df.join(
    best_by_team26.select('run_id').unique(),
    on='run_id',
    how='semi'
)
pan26_df.select('source').unique()

source
str
"""nytimes-writing-prompts-revers…"
"""nytimes-writing-prompts"""
"""nytimes-writing-prompts-tibet-…"
"""brown-cont"""
"""nytimes-writing-prompts-transl…"
"""brown"""
"""nytimes-writing-prompts-tibet-…"
"""nytimes-writing-prompts-tibet-…"


In [12]:
get_results_per_run(pan26_df_filtered_best.filter(
    pl.col('source').str.starts_with('brown')
), drop_run_and_ds=True)

team,software,roc-auc,brier,c@1,f1,f05u,mean,fpr,fnr
str,str,f64,f64,f64,f64,f64,f64,f64,f64
"""suspiciously-coherent""","""lead-log""",0.897922,0.90321,0.870098,0.722513,0.743534,0.827456,0.071429,0.31
"""turingteam""","""marinated-edge""",0.881867,0.890409,0.862889,0.602564,0.716463,0.790839,0.042208,0.45
"""sst""","""ModernBERT-large-BNN""",0.872825,0.871369,0.833333,0.704348,0.653226,0.78702,0.159091,0.19
"""odean""","""narrow-lamb""",0.767532,0.852941,0.852941,0.666667,0.714286,0.770873,0.064935,0.4
"""yjl""","""antique-impulse""",0.868961,0.844802,0.833333,0.546667,0.683333,0.755419,0.029221,0.59
…,…,…,…,…,…,…,…,…,…
"""ce-ec""","""atomic-fruit""",0.487711,0.723561,0.245098,0.393701,0.288684,0.427751,1.0,0.0
"""ikr3""","""lexical + linguistic LR""",0.846851,0.339221,0.245098,0.393701,0.288684,0.422711,1.0,0.0
"""uned-martinez""","""current-line""",0.492273,0.767011,0.711806,0.033058,0.053763,0.411582,0.068182,0.98


In [13]:
get_results_per_run(pan26_df_filtered_best.filter(
    (pl.col('source') == 'nytimes-writing-prompts-tibet-translated-en') |
    ((pl.col('source') == 'nytimes-writing-prompts') & (pl.col('label_truth') == 0))
), drop_run_and_ds=True)

team,software,roc-auc,brier,c@1,f1,f05u,mean,fpr,fnr
str,str,f64,f64,f64,f64,f64,f64,f64,f64
"""mdok""","""mdok""",0.999949,0.994874,0.994975,0.994975,0.994975,0.99595,0.005025,0.005025
"""dsgt-pan-26""","""relaxed-copy""",1.0,0.993111,0.992462,0.992405,0.996948,0.994985,0.0,0.015075
"""suspiciously-coherent""","""lead-log""",0.999747,0.985155,0.984925,0.985075,0.979228,0.986826,0.025126,0.005025
"""sst""","""ModernBERT-large-BNN""",0.999975,0.985144,0.984925,0.985149,0.976447,0.986328,0.030151,0.0
"""plagiarism-detector-com-reg2""","""alternating-contingency""",1.0,0.976415,0.987349,0.918478,0.965714,0.969591,0.0,0.015075
…,…,…,…,…,…,…,…,…,…
"""nexo""","""skinny-pigeon""",0.355799,0.493629,0.446277,0.0,0.0,0.259141,0.135678,0.994975
"""uned-martinez""","""current-line""",0.15309,0.524158,0.277038,0.151057,0.169377,0.254944,0.592965,0.849246
"""yihao""","""clear-buyer""",0.5,0.75,0.0,0.0,0.0,0.25,1.0,0.0


In [14]:
get_results_per_run(pan26_df_filtered_best.filter(
    (pl.col('source') == 'nytimes-writing-prompts-tibet-translated-en-zh-fa-en') |
    ((pl.col('source') == 'nytimes-writing-prompts') & (pl.col('label_truth') == 0))
), drop_run_and_ds=True)

team,software,roc-auc,brier,c@1,f1,f05u,mean,fpr,fnr
str,str,f64,f64,f64,f64,f64,f64,f64,f64
"""mdok""","""mdok""",0.999697,0.991744,0.98995,0.989899,0.992908,0.99284,0.005025,0.015075
"""sst""","""ModernBERT-large-BNN""",0.999848,0.984997,0.984925,0.985149,0.976447,0.986273,0.030151,0.0
"""suspiciously-coherent""","""lead-log""",0.999545,0.981349,0.984925,0.985075,0.979228,0.986024,0.025126,0.005025
"""ak-ys""","""Qwen3-14B-fine-tuned""",0.996048,0.973379,0.964824,0.964286,0.973223,0.974352,0.020101,0.050251
"""plagiarism-detector-com-reg2""","""alternating-contingency""",0.999848,0.975329,0.979293,0.930108,0.970819,0.97108,0.0,0.035176
…,…,…,…,…,…,…,…,…,…
"""zhengqiaozeng""","""three""",0.579051,0.500328,0.5,0.0,0.0,0.315876,0.0,1.0
"""nexo""","""skinny-pigeon""",0.508346,0.510816,0.45627,0.02765,0.053004,0.311217,0.135678,0.969849
"""nai-long-zhan-dui""","""solid-capital""",0.309285,0.524749,0.465329,0.027523,0.053763,0.27613,0.100503,0.969849


In [15]:
get_results_per_run(pan26_df_filtered_best.filter(
    (pl.col('source') == 'nytimes-writing-prompts-tibet-translated-mr-kk-ja-lo-iw-am-en') |
    ((pl.col('source') == 'nytimes-writing-prompts') & (pl.col('label_truth') == 0))
), drop_run_and_ds=True)

team,software,roc-auc,brier,c@1,f1,f05u,mean,fpr,fnr
str,str,f64,f64,f64,f64,f64,f64,f64,f64
"""suspiciously-coherent""","""lead-log""",0.999722,0.98156,0.987437,0.987593,0.980296,0.987322,0.025126,0.0
"""sst""","""ModernBERT-large-BNN""",0.999924,0.985017,0.984925,0.985149,0.976447,0.986292,0.030151,0.0
"""mdok""","""mdok""",0.999369,0.975389,0.967337,0.966408,0.983176,0.978336,0.005025,0.060302
"""ak-ys""","""Qwen3-14B-fine-tuned""",0.996439,0.976785,0.969849,0.969543,0.975485,0.97762,0.020101,0.040201
"""plagiarism-detector-com-reg2""","""alternating-contingency""",0.999369,0.956427,0.96543,0.875706,0.946276,0.948642,0.0,0.050251
…,…,…,…,…,…,…,…,…,…
"""latentauthors""","""roaring-assurance""",0.606121,0.500353,0.492462,0.0,0.0,0.319787,0.015075,1.0
"""original""","""central-range""",0.496957,0.548604,0.491017,0.019324,0.042918,0.319764,0.040201,0.974874
"""uned-martinez""","""current-line""",0.162913,0.539265,0.2757,0.151057,0.169377,0.259662,0.592965,0.859296


In [16]:
get_results_per_run(pan26_df_filtered_best.filter(
    (pl.col('source') == 'nytimes-writing-prompts-translated-en-obfs') |
    ((pl.col('source') == 'nytimes-writing-prompts') & (pl.col('label_truth') == 0))
), drop_run_and_ds=True)

team,software,roc-auc,brier,c@1,f1,f05u,mean,fpr,fnr
str,str,f64,f64,f64,f64,f64,f64,f64,f64
"""dsgt-pan-26""","""relaxed-copy""",1.0,0.99997,1.0,1.0,1.0,0.999994,0.0,0.0
"""mdok""","""mdok""",0.999899,0.996646,0.994975,0.994975,0.994975,0.996294,0.005025,0.005025
"""sst""","""ModernBERT-large-BNN""",0.999874,0.98496,0.984925,0.985149,0.976447,0.986271,0.030151,0.0
"""suspiciously-coherent""","""lead-log""",0.999293,0.974368,0.987437,0.987593,0.980296,0.985797,0.025126,0.0
"""plagiarism-detector-com-reg2""","""alternating-contingency""",1.0,0.972112,0.986983,0.915531,0.964409,0.967807,0.0,0.015075
…,…,…,…,…,…,…,…,…,…
"""yjl""","""antique-impulse""",0.256294,0.448123,0.427136,0.0,0.0,0.226311,0.145729,1.0
"""team-uit-huyygiaa""","""nice-frequency""",0.112131,0.494665,0.494975,0.0,0.0,0.220354,0.01005,1.0
"""nai-long-zhan-dui""","""solid-capital""",0.160274,0.48294,0.454269,0.0,0.0,0.219496,0.100503,1.0


In [17]:
get_results_per_run(pan26_df_filtered_best.filter(
    (pl.col('source') == 'nytimes-writing-prompts-reverse-swahili-translated') |
    ((pl.col('source') == 'nytimes-writing-prompts') & (pl.col('label_truth') == 0))
), drop_run_and_ds=True)

team,software,roc-auc,brier,c@1,f1,f05u,mean,fpr,fnr
str,str,f64,f64,f64,f64,f64,f64,f64,f64
"""dsgt-pan-26""","""relaxed-copy""",0.999641,0.980479,0.981221,0.833333,0.925926,0.94412,0.0,0.285714
"""mdok""","""mdok""",0.991385,0.975993,0.976526,0.8,0.862069,0.921195,0.005025,0.285714
"""sst""","""ModernBERT-large-BNN""",0.990668,0.969334,0.967136,0.787879,0.722222,0.887448,0.030151,0.071429
"""plagiarism-detector-com-reg2""","""alternating-contingency""",0.999282,0.98313,0.979898,0.444444,0.666667,0.814684,0.0,0.285714
"""turingteam""","""marinated-edge""",0.856425,0.918626,0.935264,0.482759,0.4375,0.726115,0.070352,0.357143
…,…,…,…,…,…,…,…,…,…
"""codingmates""","""silent-screen""",0.142498,0.625029,0.599661,0.024096,0.016667,0.28159,0.39196,0.928571
"""yihao""","""clear-buyer""",0.5,0.75,0.0,0.0,0.0,0.25,1.0,0.0
"""uned-martinez""","""current-line""",0.13855,0.606582,0.409795,0.03252,0.021692,0.241828,0.592965,0.857143


In [18]:
pan26_df.select('model').unique()

model
str
"""llama-3.3-70b-instruct-8bit-rt…"
"""llama-3.3-70b-instruct-8bit-wa…"
"""gpt-5.4"""
"""gpt-4o"""
"""human"""
"""gpt-5.5"""
"""nytimes-writing-prompts-transl…"


In [19]:
get_results_per_run(pan26_df_filtered_best.filter(
    ((pl.col('model') == 'gpt-5.5') | (pl.col('model') == 'human')) &
    pl.col('source').str.starts_with('brown')
), drop_run_and_ds=True)

team,software,roc-auc,brier,c@1,f1,f05u,mean,fpr,fnr
str,str,f64,f64,f64,f64,f64,f64,f64,f64
"""suspiciously-coherent""","""lead-log""",0.897922,0.90321,0.870098,0.722513,0.743534,0.827456,0.071429,0.31
"""turingteam""","""marinated-edge""",0.881867,0.890409,0.862889,0.602564,0.716463,0.790839,0.042208,0.45
"""sst""","""ModernBERT-large-BNN""",0.872825,0.871369,0.833333,0.704348,0.653226,0.78702,0.159091,0.19
"""odean""","""narrow-lamb""",0.767532,0.852941,0.852941,0.666667,0.714286,0.770873,0.064935,0.4
"""yjl""","""antique-impulse""",0.868961,0.844802,0.833333,0.546667,0.683333,0.755419,0.029221,0.59
…,…,…,…,…,…,…,…,…,…
"""ce-ec""","""atomic-fruit""",0.487711,0.723561,0.245098,0.393701,0.288684,0.427751,1.0,0.0
"""ikr3""","""lexical + linguistic LR""",0.846851,0.339221,0.245098,0.393701,0.288684,0.422711,1.0,0.0
"""uned-martinez""","""current-line""",0.492273,0.767011,0.711806,0.033058,0.053763,0.411582,0.068182,0.98


In [20]:
get_results_per_run(pan26_df_filtered_best.filter(
    ((pl.col('model') == 'gpt-5.5') | (pl.col('model') == 'human')) &
    pl.col('source').str.starts_with('nytimes')
), drop_run_and_ds=True)

team,software,roc-auc,brier,c@1,f1,f05u,mean,fpr,fnr
str,str,f64,f64,f64,f64,f64,f64,f64,f64
"""dsgt-pan-26""","""relaxed-copy""",0.999641,0.980479,0.981221,0.833333,0.925926,0.94412,0.0,0.285714
"""mdok""","""mdok""",0.991385,0.975993,0.976526,0.8,0.862069,0.921195,0.005025,0.285714
"""sst""","""ModernBERT-large-BNN""",0.990668,0.969334,0.967136,0.787879,0.722222,0.887448,0.030151,0.071429
"""plagiarism-detector-com-reg2""","""alternating-contingency""",0.999282,0.98313,0.979898,0.444444,0.666667,0.814684,0.0,0.285714
"""turingteam""","""marinated-edge""",0.856425,0.918626,0.935264,0.482759,0.4375,0.726115,0.070352,0.357143
…,…,…,…,…,…,…,…,…,…
"""codingmates""","""silent-screen""",0.142498,0.625029,0.599661,0.024096,0.016667,0.28159,0.39196,0.928571
"""yihao""","""clear-buyer""",0.5,0.75,0.0,0.0,0.0,0.25,1.0,0.0
"""uned-martinez""","""current-line""",0.13855,0.606582,0.409795,0.03252,0.021692,0.241828,0.592965,0.857143


In [21]:
get_results_per_run(pan26_df_filtered_best.filter(
    ((pl.col('model') == 'gpt-5.4') | (pl.col('model') == 'human')) &
    pl.col('source').str.starts_with('nytimes')
), drop_run_and_ds=True)

team,software,roc-auc,brier,c@1,f1,f05u,mean,fpr,fnr
str,str,f64,f64,f64,f64,f64,f64,f64,f64
"""sst""","""ModernBERT-large-BNN""",0.999937,0.993899,0.99397,0.996245,0.994006,0.995611,0.030151,0.0
"""suspiciously-coherent""","""lead-log""",0.999716,0.98314,0.992965,0.995611,0.994489,0.993184,0.025126,0.002513
"""mdok""","""mdok""",0.999754,0.986694,0.982915,0.98922,0.994898,0.990696,0.005025,0.020101
"""plagiarism-detector-com-reg2""","""alternating-contingency""",0.999804,0.963692,0.971819,0.932976,0.972067,0.968072,0.0,0.025126
"""ydong""","""arid-winch""",0.932792,0.918541,0.887978,0.934358,0.901826,0.915099,0.547739,0.007538
…,…,…,…,…,…,…,…,…,…
"""nai-long-zhan-dui""","""solid-capital""",0.403001,0.368068,0.267114,0.177329,0.334746,0.310052,0.100503,0.86809
"""aitestchecking""","""nullary-velocity""",0.777158,0.389424,0.217085,0.04884,0.112613,0.309024,0.015075,0.974874
"""zhengqiaozeng""","""three""",0.638488,0.234117,0.230151,0.072639,0.163755,0.26783,0.0,0.962312


In [22]:
get_results_per_run(pan26_df_filtered_best.filter(
    ((pl.col('model') == 'gpt-4o') | (pl.col('model') == 'human')) &
    pl.col('source').str.starts_with('nytimes')
), drop_run_and_ds=True)

team,software,roc-auc,brier,c@1,f1,f05u,mean,fpr,fnr
str,str,f64,f64,f64,f64,f64,f64,f64,f64
"""dsgt-pan-26""","""relaxed-copy""",1.0,1.0,1.0,1.0,1.0,1.0,0.0,0.0
"""spj""","""wintry-burger""",1.0,1.0,1.0,1.0,1.0,1.0,0.0,0.0
"""zhengqiaozeng""","""three""",1.0,1.0,1.0,1.0,1.0,1.0,0.0,0.0
"""plagiarism-detector-com-reg2""","""alternating-contingency""",1.0,0.999453,1.0,1.0,1.0,0.999891,0.0,0.0
"""mdok""","""mdok""",1.0,0.997637,0.997487,0.997494,0.995996,0.997723,0.005025,0.0
…,…,…,…,…,…,…,…,…,…
"""baseline""","""baseline-ppmd""",0.81806,0.775101,0.619782,0.733333,0.626186,0.714493,0.809045,0.005025
"""ce-ec""","""atomic-fruit""",0.821949,0.748857,0.5,0.666667,0.555556,0.658606,1.0,0.0
"""ikr3""","""lexical + linguistic LR""",1.0,0.564416,0.5,0.666667,0.555556,0.657328,1.0,0.0


In [23]:
get_results_per_run(pan26_df_filtered_best.filter(
    ((pl.col('model') == 'llama-3.3-70b-instruct-8bit-rtop1') | (pl.col('model') == 'human')) &
    pl.col('source').str.starts_with('nytimes')
), drop_run_and_ds=True)

team,software,roc-auc,brier,c@1,f1,f05u,mean,fpr,fnr
str,str,f64,f64,f64,f64,f64,f64,f64,f64
"""spj""","""wintry-burger""",1.0,1.0,1.0,1.0,1.0,1.0,0.0,0.0
"""dsgt-pan-26""","""relaxed-copy""",1.0,1.0,1.0,1.0,1.0,1.0,0.0,0.0
"""zhengqiaozeng""","""three""",1.0,1.0,1.0,1.0,1.0,1.0,0.0,0.0
"""plagiarism-detector-com-reg2""","""alternating-contingency""",1.0,0.998922,1.0,1.0,1.0,0.999784,0.0,0.0
"""baseline""","""baseline-binoculars-tiny-llama""",1.0,0.988863,1.0,1.0,1.0,0.997773,0.0,0.0
…,…,…,…,…,…,…,…,…,…
"""baseline""","""baseline-ppmd""",0.9933,0.69547,0.221057,0.040268,0.024793,0.394978,0.809045,0.0
"""ce-ec""","""atomic-fruit""",0.834171,0.70335,0.014851,0.029268,0.018496,0.320027,1.0,0.0
"""yihao""","""clear-buyer""",0.5,0.75,0.0,0.0,0.0,0.25,1.0,0.0


In [24]:
get_results_per_run(pan26_df_filtered_best.filter(
    ((pl.col('model') == 'llama-3.3-70b-instruct-8bit-watermark-gamma05-bias5-temp1') | (pl.col('model') == 'human')) &
    pl.col('source').str.starts_with('nytimes')
), drop_run_and_ds=True)

team,software,roc-auc,brier,c@1,f1,f05u,mean,fpr,fnr
str,str,f64,f64,f64,f64,f64,f64,f64,f64
"""dsgt-pan-26""","""relaxed-copy""",1.0,0.998187,0.997487,0.997481,0.998991,0.998429,0.0,0.005025
"""mdok""","""mdok""",1.0,0.997637,0.997487,0.997494,0.995996,0.997723,0.005025,0.0
"""team-uit-huyygiaa""","""nice-frequency""",0.998056,0.989213,0.98995,0.98995,0.98995,0.991424,0.01005,0.01005
"""suspiciously-coherent""","""lead-log""",0.999874,0.990422,0.987437,0.987593,0.980296,0.989124,0.025126,0.0
"""zhengqiaozeng""","""three""",0.999924,0.984262,0.982412,0.982097,0.992761,0.988291,0.0,0.035176
…,…,…,…,…,…,…,…,…,…
"""ce-ec""","""atomic-fruit""",0.643658,0.74675,0.497487,0.66443,0.554001,0.621265,1.0,0.005025
"""codingmates""","""silent-screen""",0.557145,0.659647,0.610982,0.552846,0.573678,0.59086,0.39196,0.356784
"""baseline""","""baseline-ppmd""",0.613368,0.735293,0.434364,0.541578,0.489591,0.562839,0.809045,0.356784


In [25]:
get_results_per_run(pan26_df_filtered_best.filter(
    ((pl.col('model') == 'nytimes-writing-prompts-translated-en-heuristic_obfs') | (pl.col('model') == 'human')) &
    pl.col('source').str.starts_with('nytimes')
), drop_run_and_ds=True)

team,software,roc-auc,brier,c@1,f1,f05u,mean,fpr,fnr
str,str,f64,f64,f64,f64,f64,f64,f64,f64
"""dsgt-pan-26""","""relaxed-copy""",1.0,0.99997,1.0,1.0,1.0,0.999994,0.0,0.0
"""mdok""","""mdok""",0.999899,0.996646,0.994975,0.994975,0.994975,0.996294,0.005025,0.005025
"""sst""","""ModernBERT-large-BNN""",0.999874,0.98496,0.984925,0.985149,0.976447,0.986271,0.030151,0.0
"""suspiciously-coherent""","""lead-log""",0.999293,0.974368,0.987437,0.987593,0.980296,0.985797,0.025126,0.0
"""plagiarism-detector-com-reg2""","""alternating-contingency""",1.0,0.972112,0.986983,0.915531,0.964409,0.967807,0.0,0.015075
…,…,…,…,…,…,…,…,…,…
"""yjl""","""antique-impulse""",0.256294,0.448123,0.427136,0.0,0.0,0.226311,0.145729,1.0
"""team-uit-huyygiaa""","""nice-frequency""",0.112131,0.494665,0.494975,0.0,0.0,0.220354,0.01005,1.0
"""nai-long-zhan-dui""","""solid-capital""",0.160274,0.48294,0.454269,0.0,0.0,0.219496,0.100503,1.0
